In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Core.logging import flow_logger as logger
from Instruments.GX_271.dim import DIM
from Instruments.Syr_pumps.HB_syr_pump import HBElite
from Instruments.Runze_valves.Runze62Valve import Runze62Valve
from Instruments.Knauer.knauer_pump_azura import KnauerPumpAzura
from Ecosystems.reaction_segment_generation import RSG
from Ecosystems.segmentation import Segmentation

In [2]:
# --- Syringe pump (RSG) ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- Runze valve ---
runze = Runze62Valve("COM8")
runze.connect()

# --- Carrier pump (Knauer) ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

HB Elite pump connected on COM10
Connected to Vici valve on COM5
Connected to Runze valve on COM8
[Runze] Valve at pos None -> state = UNKNOWN
[Runze] valve moved to pos 1 -> state = GAS_TO_DIM
Connected to Azura pump on COM4


In [3]:
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("Segmentation_testing")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=155.5,
    y_offset=8,
    x_min=145,
    x_max=250,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=319,
    y_offset=39,
    x_min=275,
    x_max=370,
    y_min=2,
    y_max=292
) 

tray.add_module(
    slot=3,
    name="dim",
    module=dim,
    x_offset=9,     #These values are to be changed
    y_offset=104,
    x_min=0,
    x_max=25,
    y_min=75,
    y_max=120
) 

In [4]:
# Instantiate the RSG ecosystem with the connected Gilson and AL1000
rsg = RSG(gilson=g, pump=syr_pump)

In [5]:
# Instantiate the segmentation ecosystem with the dim, runze valve, knauer pump and RSG connected
seg = Segmentation(dim=dim, runze=runze, carrier_pump=k_pump, rsg=rsg)

[Pump] Flow stopped
[Runze] Valve at pos 1 -> state = GAS_TO_DIM
[Runze] valve moved to pos 2 -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT


In [7]:
syr_pump.command("version")

Sent: version | Reply:

Firmware:      v3.1
Pump address:  0
Serial number: 1134722
:


['', 'Firmware:      v3.1', 'Pump address:  0', 'Serial number: 1134722', ':']

In [33]:
g.go_to_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(319.0), np.float64(106.0))

In [30]:
g.move_z(119)

'Moved Z to 119 at 125 mm/sec. Result: Move Z with Speed: 2, Success'

In [34]:
rsg.dispense_in_waste(volume = 150, rate = 0.5)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
Sent: cvolume | Reply:

:
Sent: ctvolume | Reply:

:
Sent: irate 0.5 ml/min | Reply:

:
Sent: tvolume 150 ul | Reply:

:
Sent: irun | Reply:

>


In [35]:
rsg.pickup_from_vial("rack2", 1, 150, 0.2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
Sent: cvolume | Reply:

T*
:
Sent: ctvolume | Reply:

:
Sent: wrate 0.2 ml/min | Reply:

:
Sent: tvolume 150 ul | Reply:

:
Sent: wrun | Reply:

<


In [9]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(156.0), np.float64(8.0))

In [36]:
g.move_z(127)

'Moved Z to 127 at 125 mm/sec. Result: Move Z with Speed: 2, Success'

In [20]:
print(g.current_x)
print(g.current_y)

156.0
8.0


In [37]:
def run_validation(
    rsg,
    analyte_volumes,
    start_vial_pos=2,
    n_replicates=2,
    prepickup_volume=20,   # µL
    airgap1_volume=20,     # µL - note - airgap1 seperates the working fluid and prepickup
    airgap2_volume=5,      # µL - note - airgap2 seperates the prepickup and analyte
    withdraw_rate=0.2,     # mL/min
    infuse_rate=0.5        # mL/min
):
    """
    Automated syringe pump validation workflow.

    For each analyte volume:
    - Cold start creates the full airgap1
    - Each replicate:
        - Regenerate half of airgap1 (if not first run)
        - Wash needle
        - Prepickup from wash vial (rack2, vial1)
        - Small airgap2 before analyte
        - Pickup analyte from stock (rack1, vial1)
        - Dispense everything into consecutive dilution vials
    """
    
    print("=== Starting syringe pump validation run ===")
    
    vial_pos = start_vial_pos
    half_airgap1 = airgap1_volume / 2

    # --- Cold start: create full airgap1 ---
    print(f"Cold start: taking full airgap1 ({airgap1_volume} µL)")
    rsg.take_air_gap(airgap1_volume, rate=0.05)

    for vol in analyte_volumes:
        print(f"\n=== Analyte volume: {vol} µL ===")
        
        for rep in range(1, n_replicates + 1):
            print(f"  Replicate {rep} → rack1 vial {vial_pos}")
            
            # --- Regenerate half airgap1, skip only for first replicate of first volume ---
            if not (vol == analyte_volumes[0] and rep == 1):
                rsg.take_air_gap(half_airgap1, rate=0.05)
            
            # --- Wash sequence ---
            rsg.wash_sequence()
            
            # --- Prepickup from wash vial ---
            rsg.prepickup(volume=prepickup_volume, rate=withdraw_rate)
            
            # --- Small airgap2 before analyte ---
            rsg.take_air_gap(airgap2_volume, rate=0.05)
            
            # --- Pickup analyte ---
            rsg.pickup_from_vial(
                module_name="rack1",
                vial_pos=1,
                volume=vol,
                rate=withdraw_rate
            )
            
            # --- Dispense all into dilution vial ---
            dispense_volume = vol + prepickup_volume + airgap2_volume + half_airgap1
            rsg.dispense_in_vial(
                module_name="rack1",
                vial_pos=vial_pos,
                volume=dispense_volume,
                rate=infuse_rate
            )
            
            vial_pos += 1
            
            # Optional short pause between replicates
            time.sleep(1)

    print("\n~~~ Syringe pump validation run complete ~~~")



In [38]:
analyte_volumes = [50, 40, 30, 20, 10]  # µL
n_replicates = 3                 # Means 3 runs per volume 
prepickup_volume = 20            # µL
airgap1_volume = 20              # µL - separates working fluid from prepickup
airgap2_volume = 5               # µL - separates prepickup from analyte
withdraw_rate = 0.2              # mL/min for prepickup and analyte
infuse_rate = 0.5                # mL/min for dispensing into dilution vials

run_validation(
    rsg,
    analyte_volumes=analyte_volumes,
    start_vial_pos=2,          # or wherever your first dilution vial is
    n_replicates=n_replicates,
    prepickup_volume=prepickup_volume,
    airgap1_volume=airgap1_volume,
    airgap2_volume=airgap2_volume,
    withdraw_rate=withdraw_rate,
    infuse_rate=infuse_rate
)

=== Starting syringe pump validation run ===
Cold start: taking full airgap1 (20 µL)
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
Sent: cvolume | Reply:

T*
:
Sent: ctvolume | Reply:

:
Sent: wrate 0.05 ml/min | Reply:

:
Sent: tvolume 20 ul | Reply:

:
Sent: wrun | Reply:

<

=== Analyte volume: 50 µL ===
  Replicate 1 → rack1 vial 2
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
Sent: cvolume | Reply:

T*
:
Sent: ctvolume | Reply:

:
Sent: wrate 0.05 ml/min | Reply:

:
Sent: tvolume 5.0 ul | Reply:

:
Sent: wrun | Reply:

<
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
Sent: cvolume | Reply:

T*
:
Sent: ctvolume | Reply:

:
Sent: wrate 0.5 ml/min | Reply:

:
Sent: tvolume 100.0 ul | Reply:

:
Sent: wrun | Reply:

<
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_

In [40]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


(np.float64(319.0), np.float64(106.0), 20)

In [4]:
g.home()

All axes homed successfully and Z is within safe limits


In [8]:
print(g.current_x)
print(g.current_y)

0.0
0.0


In [7]:
g.move_xy(0,0)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


'Moved to X=0, Y=0. Result: Move XY: 2, Success'

In [13]:
g.go_to_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(201.0), np.float64(106.0))

In [11]:
g.

Z below safe limit (110.00 < 130.00) — raising first.
All axes homed successfully and Z is within safe limits


In [59]:
rsg.take_air_gap(10)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [47]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(201.0), np.float64(106.0), 20)

In [53]:
g.move_z(120)

'Moved Z to 120. Result: Move Z: 2, Success'

In [12]:
g.home()

All axes homed successfully and Z is within safe limits


In [65]:
rate = 0.25          # mL/min
volume = 750           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

pump.prepare(rate=rate, volume=volume, direction=direction)

Sent: @00*PHN1
Sent: @00*RAT C 0.25 MM
Sent: @00*VOL750
Sent: @00*DIRINF


In [66]:
pump.start()

Sent: @00*RUN 1


'00I'

In [63]:
pump.stop()

Sent: @00*STP


'00P'

In [58]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(201.0), np.float64(106.0), 20)

In [10]:
g.move_z(130)

'Moved Z to 130. Result: Move Z: 2, Success'

In [79]:
rsg.take_air_gap(10)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [80]:
rsg.wash_sequence()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL100.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL105.0
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP


In [73]:
rsg.prepickup()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [74]:
rsg.take_air_gap(5)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [19]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(36.0), np.float64(8.0))

In [6]:
g.move_z(60)

'Moved Z to 60. Result: No valid response received.'

In [75]:
rsg.pickup_from_vial("rack1", 1, 4)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL4
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [76]:
g.go_to_vial("rack1", 25)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(52.55), np.float64(159.5295))

In [77]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [78]:
rsg.dispense_in_vial("rack1", 25, 29)

#Note - analyte vol + 5 + 10 + 10 (final 10 is 10uL of the 40uL air gap, leaving 30uL,
# this 10uL is reintroduced before the wash sequence)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL29
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP
